# Phase 8 — Market Basket Analysis

Goal: discover products that are frequently purchased together and generate
simple cross-sell recommendations using association rules.

In [1]:
# Section 1 — Install compatible dependency

# Run once if mlxtend import gives a pandas compatibility error.
%pip install -q --upgrade "numpy<2" "mlxtend==0.23.4" "scikit-learn>=1.3" "scipy>=1.10" pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\ProgramData\anaconda3\python.exe -m pip install --upgrade pip


In [2]:
# Section 2 — Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from mlxtend.frequent_patterns import apriori, association_rules

C:\Users\arman\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\arman\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [3]:
# Section 3 — Load sales data

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

SALES_PATH = PROCESSED_DATA_DIR / "customer_sales.parquet"

sales_df = pd.read_parquet(SALES_PATH)

print("Sales data:", sales_df.shape)
print("Columns:", sales_df.columns.tolist())

Sales data: (779425, 12)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'LineTotal', 'IsCancelled', 'IsReturn', 'Revenue']


In [4]:
# Section 4 — Prepare transactions

basket_data = sales_df[
    ["Invoice", "StockCode", "Description"]
].dropna(subset=["Invoice", "StockCode"])

# Remove non-product/service codes identified during EDA.
non_product_codes = [
    "M",
    "POST",
    "C2",
    "DOT",
    "BANK CHARGES",
    "D"
]

basket_data = basket_data[
    ~basket_data["StockCode"].isin(non_product_codes)
].copy()

# One product should appear at most once per invoice.
basket_data = basket_data.drop_duplicates(
    subset=["Invoice", "StockCode"]
)

print("Transaction-product rows:", basket_data.shape)
print("Invoices:", basket_data["Invoice"].nunique())
print("Products:", basket_data["StockCode"].nunique())

Transaction-product rows: (766143, 3)
Invoices: 36639
Products: 4625


In [5]:
# Section 5 — Product lookup

product_lookup = (
    basket_data[
        ["StockCode", "Description"]
    ]
    .drop_duplicates("StockCode")
    .set_index("StockCode")["Description"]
)

print("Product lookup:", product_lookup.shape)

Product lookup: (4625,)


In [6]:
# Section 6 — Transaction matrix

basket = (
    basket_data
    .assign(value=1)
    .pivot_table(
        index="Invoice",
        columns="StockCode",
        values="value",
        fill_value=0,
        aggfunc="max"
    )
    .astype(bool)
)

print("Basket shape:", basket.shape)

Basket shape: (36639, 4625)


In [ ]:
# Section 7 — Frequent itemsets

# 1% support keeps the analysis focused on repeatable product combinations.
frequent_itemsets = apriori(
    basket,
    min_support=0.01,
    use_colnames=True
)

frequent_itemsets["item_count"] = (
    frequent_itemsets["itemsets"].apply(len)
)

frequent_itemsets = (
    frequent_itemsets
    .sort_values("support", ascending=False)
    .reset_index(drop=True)
)

print("Frequent itemsets:", len(frequent_itemsets))

display(
    frequent_itemsets.head(20)
)

In [ ]:
# Section 8 — Association rules

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.30,
    num_itemsets=len(basket)
)

# Keep simple one-product -> one-product rules for easy business interpretation.
rules = rules[
    (rules["antecedents"].apply(len) == 1) &
    (rules["consequents"].apply(len) == 1)
].copy()

rules = rules[
    rules["lift"] > 1
].copy()

rules = (
    rules
    .sort_values(
        ["lift", "confidence", "support"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("Useful rules:", len(rules))

display(
    rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20)
)

In [10]:
# Section 9 — Make rules human-readable

def item_name(itemset):
    code = next(iter(itemset))
    description = product_lookup.get(code, "Unknown product")
    return f"{code} — {description}"

rules["antecedent_product"] = rules["antecedents"].apply(item_name)
rules["consequent_product"] = rules["consequents"].apply(item_name)

readable_rules = rules[
    [
        "antecedent_product",
        "consequent_product",
        "support",
        "confidence",
        "lift"
    ]
].copy()

readable_rules["support_pct"] = (
    readable_rules["support"] * 100
).round(2)

readable_rules["confidence_pct"] = (
    readable_rules["confidence"] * 100
).round(2)

readable_rules["lift"] = readable_rules["lift"].round(2)

readable_rules = readable_rules[
    [
        "antecedent_product",
        "consequent_product",
        "support_pct",
        "confidence_pct",
        "lift"
    ]
]

display(readable_rules.head(20))

NameError: name 'rules' is not defined

In [11]:
# Section 10 — Top recommendations by lift

top_rules = (
    readable_rules
    .sort_values(
        ["lift", "confidence_pct"],
        ascending=False
    )
    .head(20)
)

display(top_rules)

NameError: name 'readable_rules' is not defined

In [12]:
# Section 11 — Basic rule quality summary

print("Number of rules:", len(readable_rules))

if len(readable_rules) > 0:
    print("Highest lift:", readable_rules["lift"].max())
    print("Highest confidence (%):", readable_rules["confidence_pct"].max())
    print("Average lift:", round(readable_rules["lift"].mean(), 2))
else:
    print("No rules met the current thresholds.")

NameError: name 'readable_rules' is not defined

In [13]:
# Section 12 — Save outputs

FREQUENT_ITEMSETS_PATH = (
    PROCESSED_DATA_DIR /
    "market_basket_frequent_itemsets.parquet"
)

RULES_PATH = (
    PROCESSED_DATA_DIR /
    "market_basket_rules.parquet"
)

frequent_itemsets.to_parquet(
    FREQUENT_ITEMSETS_PATH,
    index=False
)

# Save the human-readable rule table.
readable_rules.to_parquet(
    RULES_PATH,
    index=False
)

print("Saved:", FREQUENT_ITEMSETS_PATH)
print("Saved:", RULES_PATH)

NameError: name 'frequent_itemsets' is not defined

In [14]:
# Section 13 — Final validation

print("Market Basket Analysis completed.")
print("Frequent itemsets:", len(frequent_itemsets))
print("Association rules:", len(readable_rules))
print("Frequent itemsets file:", FREQUENT_ITEMSETS_PATH)
print("Rules file:", RULES_PATH)

Market Basket Analysis completed.


NameError: name 'frequent_itemsets' is not defined